In [30]:
from typing import Iterable

from ortools.sat.python import cp_model


class SolutionCallback(cp_model.CpSolverSolutionCallback):

    def __init__(self, variables: Iterable[cp_model.IntVar]) -> None:
        super().__init__()
        self.variables = list(variables)

    def on_solution_callback(self) -> None:
        for var in self.variables:
            print(f"{var.name}: {self.boolean_value(var)}")


model = cp_model.CpModel()
bool_var = model.NewBoolVar("x")
solution_callback = SolutionCallback([bool_var])
solver = cp_model.CpSolver()
solver.parameters.enumerate_all_solutions = True
solver.Solve(model, solution_callback)

x: False
x: True


4

In [33]:
from functools import cache
from itertools import count
from typing import Generic, Iterable, Literal, Mapping, Protocol, TypeVar

from ortools.sat.python import cp_model
from pydantic import BaseModel


var_ids = count()


class Var(Protocol):
    def __hash__(self) -> int: ...
    def _int_vars(self, model: cp_model.CpModel) -> list[cp_model.IntVar]: ...
    def _transform(self, values: Iterable[int]) -> bool | str: ...


class BoolVar(Var):
    def __hash__(self) -> int:
        return id(self)

    @cache
    def _int_vars(self, model: cp_model.CpModel) -> list[cp_model.IntVar]:
        return [model.new_bool_var(str(next(var_ids)))]

    def _transform(self, values: Iterable[int]) -> bool | str:
        return bool(list(values)[0])


T = TypeVar("T", bound=bool | str)


class CategoricalVar(Var, Generic[T]):
    def __init__(self, values: Iterable[T]) -> None:
        self._values = list(values)

    def __hash__(self) -> int:
        return id(self)

    @cache
    def _int_vars(self, model: cp_model.CpModel) -> list[cp_model.IntVar]:
        int_vars = [model.new_bool_var(str(next(var_ids))) for _ in self._values]
        model.add_exactly_one(int_vars)
        return int_vars

    def _transform(self, values: Iterable[int]) -> bool | str:
        return self._values[list(values).index(1)]

    def __eq__(self, other: T) -> "Constraint":
        @cache
        def constraint(model: cp_model.CpModel) -> cp_model.IntVar:
            return self._int_vars(model)[self._values.index(other)]

        return constraint

    def __ne__(self, other: T) -> "Constraint":
        @cache
        def constraint(model: cp_model.CpModel) -> cp_model.IntVar:
            var = self._int_vars(model)[self._values.index(other)]
            neg_var = model.new_bool_var(str(next(var_ids)))
            model.add(var == 0).only_enforce_if(neg_var)
            model.add(var == 1).only_enforce_if(~neg_var)
            return neg_var

        return constraint


class Constraint(Protocol):
    def __call__(self, model: cp_model.CpModel) -> cp_model.IntVar: ...


def get_solutions(
    vars: Mapping[str, Var], constraints: Iterable[Constraint]
) -> list[Mapping[str, bool | str]]:
    model = cp_model.CpModel()
    solutions: list[Mapping[str, bool | str]] = []
    int_vars_and_transforms = {
        name: (var._int_vars(model), var._transform) for name, var in vars.items()
    }
    for constraint in constraints:
        model.add(constraint(model) == 1)

    class SolutionCallback(cp_model.CpSolverSolutionCallback):
        def on_solution_callback(self) -> None:
            solutions.append(
                {
                    name: transform(self.value(int_var) for int_var in int_vars)
                    for name, (
                        int_vars,
                        transform,
                    ) in int_vars_and_transforms.items()
                }
            )

    solver = cp_model.CpSolver()
    solver.parameters.enumerate_all_solutions = True
    solver.Solve(model, SolutionCallback())
    return solutions


Y = Literal["a", "b", "c"]


class SolutionVars(BaseModel):
    x: BoolVar = BoolVar()
    y: CategoricalVar[Y] = CategoricalVar(["a", "b", "c"])

    model_config = {"arbitrary_types_allowed": True}


class Solution(BaseModel):
    x: bool
    y: Y


vars = SolutionVars()

solutions = [
    Solution.model_validate(solution)
    for solution in get_solutions(vars.model_dump(), [vars.y != "a"])
]

In [26]:
SolutionVars().model_dump()

{'x': <__main__.BoolVar at 0x113d49cd0>,
 'y': <__main__.CategoricalVar at 0x113d49f40>}

In [ ]:
from dataclasses import dataclass
from pydantic import BaseModel, Field


def bool_var() -> bool: ...


class Solution(BaseModel):
    x: bool = bool_var()


Solution.x

AttributeError: x

In [ ]:
class Var: ...

In [ ]:
from typing import Generic, TypeVar


class Types:
    bool: bool


T = TypeVar("T", bound=Types)


class Game(Generic[T]):
    murderer: T.bool

In [64]:
from dataclasses import dataclass
from pydantic import BaseModel, Field


class MyBool:
    @property
    def value(self) -> bool:
        return Field(default_factory=bool)


class Solution(BaseModel):
    x: bool = MyBool().value


Solution()

Solution(x=False)

In [47]:
class MyBool:
    def __bool__(self) -> bool:
        return MyBool()


any([MyBool(), MyBool()])

TypeError: __bool__ should return bool, returned MyBool

In [7]:
bool_var.name

'x'